# Step 5: compare the explanations

Compares the SHAP gene rankings of pairs of nodes (thesis Section 3.6), with all nodes and without quiet nodes, and
saves the tables to `runs/results/`. The statistical tests and figures are in `06_tests_and_figures`. Takes about 10
to 20 minutes.

Settings.

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's startup messages

PROJECT = os.path.expandvars("$SCRATCH/CSC-488-Code")    # project folder: data, lab code and results
DATA_PATH = os.path.join(PROJECT, "data", "Copy of ecoli_MG1655plus_genome_part2_log_counts_norm_01.csv")
SEQADAGE = os.path.join(PROJECT, "seqADAGE")              # the lab's code (my fork, Jordin branch)
RUN_DIR = os.path.join(PROJECT, "runs")                   # every notebook saves and reads its results here
os.chdir(PROJECT)

N_GROUPS, MODELS_PER_GROUP, BASE_SEED = 5, 25, 171       # 5 groups of 25 models, seeds 171 to 295

import itertools
import time
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from scipy.stats import rankdata

TOP_K = 50                   # for the top-50 overlap
MAX_PAIRS_PER_CLUSTER = 500  # same-cluster pairs per cluster, at most
N_SINGLE_PAIRS = 100         # pairs of single models from different groups
QUIET = 0.15                 # a node is quiet if its weight magnitude is below this
MAX_CLUSTER_SIZE = 100       # regular clusters have 2 to 100 nodes
SEED = 0                     # for picking random pairs

File paths, and loading the saved results.

In [ ]:
def group_seed(g, i):
    """Seed of model i of group g (model number N = 25 g + i has seed 171 + N)."""
    return BASE_SEED + g * MODELS_PER_GROUP + i


def model_path(g, seed):
    return os.path.join(RUN_DIR, "models", f"group{g}", f"model_seed{seed}.npz")


def eadage_path(g):
    return os.path.join(RUN_DIR, "eadage", f"eadage_group{g}.npz")


def load_model(path):
    """A saved model (single model or eADAGE) as a dict of arrays."""
    with np.load(path, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}


def shap_path(g, seed=None):
    """SHAP importance (genes x nodes) of a single model, or of group g's eADAGE with seed=None."""
    return os.path.join(RUN_DIR, "shap", f"group{g}", "shap_eadage.npy" if seed is None else f"shap_seed{seed}.npy")

Weighted Pearson correlation between nodes, the same calculation as the lab's script but written with matrix products
so it is fast, and the distance (1 - r) / 2 (thesis Equation 5).

In [ ]:
def weighted_pearson_corr(Wa, Wb, block=500):
    """Correlation between every column (node) of Wa and every column of Wb. Each gene counts in proportion to
    the average of the two nodes' absolute weights."""
    Wa, Wb = np.asarray(Wa, "float64"), np.asarray(Wb, "float64")
    Ab, AWb, WWb = np.abs(Wb), np.abs(Wb) * Wb, Wb * Wb
    sb, pb, qb = Ab.sum(0), AWb.sum(0), (Ab * WWb).sum(0)
    R = np.empty((Wa.shape[1], Wb.shape[1]), "float32")
    for s in range(0, Wa.shape[1], block):
        X = Wa[:, s:s + block]
        A, AX, XX = np.abs(X), np.abs(X) * X, X * X
        sw = 0.5 * (A.sum(0)[:, None] + sb[None, :])         # total weight
        mx = 0.5 * (AX.sum(0)[:, None] + X.T @ Ab) / sw      # weighted mean of node a
        my = 0.5 * (A.T @ Wb + pb[None, :]) / sw             # weighted mean of node b
        cov = 0.5 * (AX.T @ Wb + X.T @ AWb) / sw - mx * my
        vx = np.maximum(0.5 * ((A * XX).sum(0)[:, None] + XX.T @ Ab) / sw - mx ** 2, 1e-30)
        vy = np.maximum(0.5 * (A.T @ WWb + qb[None, :]) / sw - my ** 2, 1e-30)
        R[s:s + block] = np.clip(cov / np.sqrt(vx * vy), -1, 1)
    return R


def weighted_pearson_dist(Wa, Wb):
    return (1 - weighted_pearson_corr(Wa, Wb)) / 2

`NodeTable` holds every node's gene ranks and top-50 genes, so that many node pairs can be compared at once: the
Spearman correlation of two nodes' importances, and the fraction of their top-50 genes they share.

In [ ]:
def standardize(M):
    """Center and scale every column, so the correlation of two columns is the mean of their product."""
    M = np.asarray(M, "float64")
    M = M - M.mean(0)
    sd = M.std(0)
    sd[sd == 0] = np.nan
    return (M / sd).astype("float32")


class NodeTable:
    def __init__(self, Ws, imps):
        W, imp = np.hstack(Ws), np.hstack(imps)
        self.ranks = standardize(rankdata(imp, axis=0))     # the mean product of two columns is their Spearman
        top = np.argpartition(-imp, TOP_K, axis=0)[:TOP_K]  # each node's top-50 genes by importance
        self.top = np.zeros(imp.shape, dtype=bool)
        np.put_along_axis(self.top, top, True, axis=0)
        self.weight_norm = np.linalg.norm(W, axis=0)

    def corr(self, other, ia, ib, chunk=5000):
        """Compare node ia[k] of this table with node ib[k] of `other`, for every k."""
        sp, top = [], []
        for s in range(0, len(ia), chunk):
            a, b = ia[s:s + chunk], ib[s:s + chunk]
            sp.append((self.ranks[:, a] * other.ranks[:, b]).mean(0))
            top.append((self.top[:, a] & other.top[:, b]).sum(0) / TOP_K)
        return {"shap_spearman": np.concatenate(sp), "top_overlap": np.concatenate(top)}


def pair_table(comparison, corrs, **cols):
    df = pd.DataFrame(corrs)
    df.insert(0, "comparison", comparison)
    for k, v in cols.items():
        df[k] = v
    return df

Within one group: pairs of nodes from different models in the same cluster (at most 500 per cluster), as many random
pairs from different clusters (chance level), and each single-model node against its own eADAGE node. `keep` marks
the pairs used without quiet nodes.

In [ ]:
def same_cluster_pairs(members, nm, rng):
    """Pairs of nodes in one cluster from different models: all of them, or MAX_PAIRS_PER_CLUSTER at random."""
    if len(members) * (len(members) - 1) // 2 <= MAX_PAIRS_PER_CLUSTER:
        return [(a, b) for a, b in itertools.combinations(members, 2) if nm[a] != nm[b]]
    a, b = rng.choice(members, size=(2, 4 * MAX_PAIRS_PER_CLUSTER))
    keep = nm[a] != nm[b]
    pairs = sorted(set(zip(np.minimum(a, b)[keep].tolist(), np.maximum(a, b)[keep].tolist())))
    return [pairs[k] for k in rng.permutation(len(pairs))[:MAX_PAIRS_PER_CLUSTER]]


def within_group(g, rng):
    """Node pairs within group g, and which of the group's eADAGE nodes come from regular clusters."""
    seeds = [group_seed(g, i) for i in range(MODELS_PER_GROUP)]
    models = [load_model(model_path(g, s)) for s in seeds]
    e = load_model(eadage_path(g))
    ind = NodeTable([m["W"] for m in models], [np.load(shap_path(g, s)) for s in seeds])
    ens = NodeTable([e["W"]], [np.load(shap_path(g))])
    labels, nm, ni = e["labels"], e["node_model"], e["node_idx"]  # each single-model node's cluster, model, number
    seed_of = np.array(seeds)[nm]
    K = e["W"].shape[1]
    sizes = np.bincount(labels, minlength=K)
    member_norm = np.array([np.median(ind.weight_norm[labels == c]) if sizes[c] else np.nan for c in range(K)])
    quiet = ind.weight_norm < QUIET
    regular = (sizes > 1) & (sizes <= MAX_CLUSTER_SIZE) & (member_norm >= QUIET)

    n = np.arange(len(labels))
    to_e = pair_table("ind_vs_eadage", ind.corr(ens, n, labels), group=g, seed_a=seed_of, seed_b=-1, node_a=ni,
                      node_b=labels, cluster=labels, cluster_size=sizes[labels], keep=regular[labels] & ~quiet)

    pairs = []
    for c in np.where(sizes > 1)[0]:
        pairs += same_cluster_pairs(np.where(labels == c)[0], nm, rng)
    ia, ib = np.array(pairs, dtype=int).reshape(-1, 2).T
    same = pair_table("ind_vs_ind", ind.corr(ind, ia, ib), group=g, seed_a=seed_of[ia], seed_b=seed_of[ib],
                      node_a=ni[ia], node_b=ni[ib], cluster=labels[ia], cluster_size=sizes[labels[ia]],
                      keep=regular[labels[ia]] & ~quiet[ia] & ~quiet[ib])

    a, b = rng.integers(len(labels), size=(2, 4 * len(ia) + 100))
    ok = (nm[a] != nm[b]) & (labels[a] != labels[b])
    a, b = a[ok][:len(ia)], b[ok][:len(ia)]
    a, b = np.where(nm[a] < nm[b], a, b), np.where(nm[a] < nm[b], b, a)  # lower model first
    null = pair_table("random_pairs", ind.corr(ind, a, b), group=g, seed_a=seed_of[a], seed_b=seed_of[b],
                      node_a=ni[a], node_b=ni[b], cluster=-1, cluster_size=-1,
                      keep=regular[labels[a]] & regular[labels[b]] & ~quiet[a] & ~quiet[b])
    print(f"group {g}: {len(same):,} same-cluster pairs, {int(quiet.sum()):,} quiet nodes, "
          f"{int(regular.sum())} of {K} clusters regular")
    return pd.concat([to_e, same, null], ignore_index=True), regular

Across groups: match every node of one model to a different node of another with the Hungarian algorithm on the
weighted Pearson distance, and compare the matched pairs and a random other node (the chance-level control). Done for
all 10 pairs of eADAGEs and for 100 random pairs of single models from different groups.

In [ ]:
def compare_matched(label, Wa, Wb, imp_a, imp_b, keep_a, keep_b, rng, **cols):
    ta, tb = NodeTable([Wa], [imp_a]), NodeTable([Wb], [imp_b])
    r, c = linear_sum_assignment(weighted_pearson_dist(Wa, Wb))
    match = c[np.argsort(r)]
    K = Wb.shape[1]
    ia = np.arange(Wa.shape[1])
    other = (match + 1 + rng.integers(K - 1, size=len(match))) % K  # any node of b except the match
    return pd.concat([pair_table(label, ta.corr(tb, ia, match), node_a=ia, node_b=match,
                                 keep=keep_a & keep_b[match], **cols),
                      pair_table(label + "_random", ta.corr(tb, ia, other), node_a=ia, node_b=other,
                                 keep=keep_a & keep_b[other], **cols)], ignore_index=True)


def across_groups(regular, rng):
    out = []
    for ga, gb in itertools.combinations(range(N_GROUPS), 2):
        ea, eb = load_model(eadage_path(ga)), load_model(eadage_path(gb))
        out.append(compare_matched("across_eadage", ea["W"], eb["W"], np.load(shap_path(ga)), np.load(shap_path(gb)),
                                   regular[ga], regular[gb], rng, group=ga, group_b=gb, seed_a=-1, seed_b=-1))
    all_pairs = [(ga, ia, gb, ib) for ga, gb in itertools.combinations(range(N_GROUPS), 2)
                 for ia in range(MODELS_PER_GROUP) for ib in range(MODELS_PER_GROUP)]
    pick = rng.choice(len(all_pairs), N_SINGLE_PAIRS, replace=False)
    not_quiet = lambda W: np.linalg.norm(W, axis=0) >= QUIET
    for ga, ia, gb, ib in (all_pairs[p] for p in sorted(pick)):
        sa, sb = group_seed(ga, ia), group_seed(gb, ib)
        Wa, Wb = load_model(model_path(ga, sa))["W"], load_model(model_path(gb, sb))["W"]
        out.append(compare_matched("across_ind", Wa, Wb, np.load(shap_path(ga, sa)), np.load(shap_path(gb, sb)),
                                   not_quiet(Wa), not_quiet(Wb), rng, group=ga, group_b=gb, seed_a=sa, seed_b=sb))
    return pd.concat(out, ignore_index=True)

Summaries: the median over each pair of models' node pairs, then over all node pairs.

In [ ]:
METRICS = ["shap_spearman", "top_overlap"]
ORDER = ["random_pairs", "ind_vs_ind", "ind_vs_eadage", "across_ind_random", "across_ind", "across_eadage_random",
         "across_eadage"]


def summarize(nodes):
    """(one row per pair of models, one row per comparison over all node pairs)."""
    per_pair = nodes.groupby(["comparison", "group", "group_b", "seed_a", "seed_b"], dropna=False)[METRICS] \
                    .median().reset_index()
    overall = pd.DataFrame([{"comparison": c, "node_pairs": int((nodes["comparison"] == c).sum()),
                             "model_pairs": int((per_pair["comparison"] == c).sum()),
                             **{f"{m}_median": nodes.loc[nodes["comparison"] == c, m].median() for m in METRICS}}
                            for c in ORDER])
    return per_pair, overall

Run everything and save the tables to `runs/results/`.

In [ ]:
start = time.time()
rng = np.random.default_rng(SEED)
parts, regular = [], {}
for g in range(N_GROUPS):
    nodes_g, regular[g] = within_group(g, rng)
    parts.append(nodes_g)
parts.append(across_groups(regular, rng))
nodes = pd.concat(parts, ignore_index=True)

per_pair, overall = summarize(nodes)
per_pair_kept, overall_kept = summarize(nodes[nodes["keep"]])

out = os.path.join(RUN_DIR, "results")
os.makedirs(out, exist_ok=True)
for name, df in [("node_pairs", nodes), ("model_pairs", per_pair), ("summary", overall),
                 ("model_pairs_without_quiet", per_pair_kept), ("summary_without_quiet", overall_kept)]:
    df.to_csv(os.path.join(out, f"{name}.csv"), index=False)
print(f"done in {(time.time() - start) / 60:.0f} min")

Median agreement over all node pairs, with all nodes and without quiet nodes.

In [ ]:
overall.set_index("comparison").join(overall_kept.set_index("comparison")[[f"{m}_median" for m in METRICS]],
                                     rsuffix="_without_quiet").round(3)